In [ ]:
import pandas as pd
from ete3 import Tree
from collections import deque
from pathlib import Path

# Set up project paths
project_root = Path('/workspaces/CellTreeBench')
data_dir = project_root / 'data' / 'celegans_small' / 'raw'
output_dir = project_root / 'data' / 'celegans_small' / 'P0' / 'tree_building'

# Ensure output directory exists
output_dir.mkdir(parents=True, exist_ok=True)

# Load the data
file_name = data_dir / "CellTable.csv"
cell_table = pd.read_csv(file_name)
print(f"Loaded CellTable from: {file_name}")
print(f"Output directory: {output_dir}")

# Identify root lineages
root_lineages = cell_table[
    cell_table["ProgenitorDatasetName"].notna() & cell_table["ParentDatasetName"].isna()
][["Lineage", "ProgenitorDatasetName", "Parent"]]
root_lineages = root_lineages[~root_lineages["ProgenitorDatasetName"].str.contains("/")]
root_lineages = root_lineages.reset_index(drop=True)

# Function to find the root of the lineage tree based on the Molecular Tree
def get_lineage_descent(cell_table, root_molecular):
    cur_lineage = "ProgenitorDatasetName"
    par_lineage = "ParentDatasetName"
    res = []
    for index, row in cell_table.iterrows():
        if row[cur_lineage] == root_molecular:
            res.append(row)

    par_list = [root_molecular]
    while len(par_list) > 0:
        the_par = par_list.pop(0)
        for index, row in cell_table.iterrows():
            if row[par_lineage] == the_par:
                if pd.isna(row[cur_lineage]) or "/" in row[cur_lineage]:
                    continue
                res.append(row)
                if row[cur_lineage] not in par_list:
                    par_list.append(row[cur_lineage])
    return pd.DataFrame(res)

# Function to create a tree from the DataFrame
def create_tree(tree_df, the_level):
    # subset tree_df by level
    tree_df_selected = tree_df[tree_df["Level"] <= the_level]

    # Create the root of the tree
    node_dict = {}
    root_name = tree_df_selected[tree_df_selected["Level"] == 0]["Lineage"].values[0]
    tree = Tree(name=root_name)
    node_dict[root_name] = tree

    # Create nodes and arrange by parent
    for index, row in tree_df_selected.iterrows():
        lineage_name = row["Lineage"]
        parent_name = row["Parent"]
        if lineage_name == root_name:
            continue
        if lineage_name not in node_dict:
            node_dict[lineage_name] = Tree(name=lineage_name)
        if parent_name not in node_dict:
            print(f"ERR: Parent node {parent_name} not found for {lineage_name}.")
        else:
            node_dict[parent_name].add_child(node_dict[lineage_name])
    return tree

# Iterate through each root molecular tree and compute the number of leaves
leaf_counts = {}
for root_molecular_tree in root_lineages["ProgenitorDatasetName"].unique():
    df = get_lineage_descent(cell_table, root_molecular_tree)
    
    # Create the lineage tree DataFrame
    tree_df = df[["ProgenitorDatasetName", "ParentDatasetName"]].drop_duplicates()
    tree_df.columns = ["Lineage", "Parent"]
    tree_df["Level"] = 0

    # Create a dictionary for parent lookup from the DataFrame
    parent_dict = tree_df.set_index("Lineage")["Parent"].to_dict()

    # Initialize the level dictionary and queue for BFS
    level_dict = {}
    queue = deque()

    for lineage, parent in parent_dict.items():
        if pd.isna(parent):
            queue.append((lineage, 0))

    # BFS to calculate levels
    while queue:
        current_node, current_level = queue.popleft()
        level_dict[current_node] = current_level

        # Find all children of the current node
        children = [
            lineage for lineage, parent in parent_dict.items() if parent == current_node
        ]
        for child in children:
            if child not in level_dict:  # To avoid processing the same node twice
                queue.append((child, current_level + 1))
    
    # Assign levels to the DataFrame
    tree_df["Level"] = tree_df["Lineage"].map(level_dict)

    # Create the molecular tree
    molecular_tree = create_tree(tree_df, tree_df["Level"].max())

    # Count the number of leaves
    leaf_count = len(molecular_tree.get_leaf_names())
    leaf_counts[root_molecular_tree] = leaf_count

    print(f"Root Molecular Tree: {root_molecular_tree}, Number of Leaves: {leaf_count}")
    print(molecular_tree.get_ascii(attributes=["name"]))

# Print the leaf counts for all root molecular trees
print("Leaf counts for each root molecular tree:")
print(leaf_counts)

Root Molecular Tree: ABaxx, Number of Leaves: 11

                          /ABarpaaaa-ABarpaaaaa
                  /ABarpaaa
           /ABarpaa       \-ABarpaaap
          |      |
          |       \ABarpaapABarpaapx-ABarpaapaa
          |
          |--ABarpax
          |
-ABaxxABarpx                       /-ABarppxaaa
          |               /ABarppxaa
          |              |         \-ABarppxaap
          |       /ABarppxa
          |      |       |         /ABarppxapp-ABarppxappx
          |      |        \ABarppxap
           \ABarppx                \-ABarppxapa
                 |
                 |        /ABarppxpa-ABarppxpax
                  \ABarppxp
                         |         /-ABarppxppa
                          \ABarppxpp
                                   \-ABarppxppp
Root Molecular Tree: ABalapx, Number of Leaves: 6

       /-ABalapaa
      |
      |        /-ABalapapa
      |       |
      |       |         /-ABalapxppa
-ABalapxABalapxpABalapxpp
      | 

In [2]:
# Print the leaf counts for all root molecular trees
print("Leaf counts for each root molecular tree:")
print(leaf_counts)

Leaf counts for each root molecular tree:
{'ABaxx': 11, 'ABalapx': 6, 'ABalpax': 1, 'ABalppa': 1, 'ABaraax': 7, 'ABarapa': 1, 'ABpxax': 27, 'ABpxp': 39, 'Cx': 16, 'Dx': 5, 'Exx': 4, 'MSx': 30}
